<a href="https://colab.research.google.com/github/Nadsyuhamus/Databladez_Tourism/blob/feature%2Fml-tourism-engine/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Tourism Data Audit

## Purpose

Understand the available tourism datasets before selecting features or training a model.

## Questions

1. What does each dataset represent?
2. What are the geographic and temporal coverage?
3. Which columns contain missing values?
4. Which variables can legitimately be used for forecasting?
5. Which data is state-level and which is district-level?
6. Is there data leakage or duplicated information?
7. What prediction target can be defended?

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/Nadsyuhamus/Databladez_Tourism.git"
BRANCH = "feature/ml-tourism-engine"
REPO_DIR = Path("/content/Databladez_Tourism")

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin {BRANCH}

Cloning into '/content/Databladez_Tourism'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 124 (delta 47), reused 71 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 317.97 KiB | 4.75 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [5]:
DATA_DIR = (
    REPO_DIR
    / "outputs"
    / "datathon_cleaned_all_files"
)

print("Correct data directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

Correct data directory: /content/Databladez_Tourism/outputs/datathon_cleaned_all_files
Directory exists: True


In [6]:
required_files = [
    "cleaned_data.csv",
    "state_tourism.csv",
    "district_context.csv",
    "google_trends_monthly.csv",
    "destination_rankings_2024_2025.csv",
    "data_dictionary.csv",
]

missing_files = []

for filename in required_files:
    file_path = DATA_DIR / filename

    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"✓ {filename:<40} {size_mb:.2f} MB")
    else:
        missing_files.append(filename)
        print(f"✗ {filename}")

if missing_files:
    raise FileNotFoundError(
        f"Missing required files: {missing_files}"
    )

print("\nAll required datasets are available.")

✓ cleaned_data.csv                         0.03 MB
✓ state_tourism.csv                        0.02 MB
✓ district_context.csv                     0.08 MB
✓ google_trends_monthly.csv                1.15 MB
✓ destination_rankings_2024_2025.csv       0.02 MB
✓ data_dictionary.csv                      0.00 MB

All required datasets are available.


In [8]:
import pandas as pd
import numpy as np
from IPython.display import display

file_paths = {
    "cleaned_data": DATA_DIR / "cleaned_data.csv",
    "state_tourism": DATA_DIR / "state_tourism.csv",
    "district_context": DATA_DIR / "district_context.csv",
    "google_trends": DATA_DIR / "google_trends_monthly.csv",
    "destination_rankings": DATA_DIR / "destination_rankings_2024_2025.csv",
    "data_dictionary": DATA_DIR / "data_dictionary.csv",
}

datasets = {
    name: pd.read_csv(path)
    for name, path in file_paths.items()
}

print("Datasets loaded successfully:\n")

for name, dataframe in datasets.items():
    print(f"{name:<25} {dataframe.shape[0]:>6} rows × "
          f"{dataframe.shape[1]:>2} columns")

Datasets loaded successfully:

cleaned_data                 112 rows × 27 columns
state_tourism                112 rows × 15 columns
district_context             960 rows × 19 columns
google_trends              14810 rows ×  8 columns
destination_rankings         320 rows ×  5 columns
data_dictionary               27 rows ×  5 columns


In [9]:
audit_summary = []

for name, dataframe in datasets.items():
    total_cells = dataframe.shape[0] * dataframe.shape[1]
    missing_cells = int(dataframe.isna().sum().sum())

    audit_summary.append({
        "dataset": name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1],
        "duplicate_rows": int(dataframe.duplicated().sum()),
        "missing_cells": missing_cells,
        "missing_percentage": (
            round(missing_cells / total_cells * 100, 2)
            if total_cells > 0
            else 0
        ),
    })

audit_summary = pd.DataFrame(audit_summary)

display(audit_summary)

,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
0,cleaned_data,112,27,0,425,14.05
1,state_tourism,112,15,0,0,0.00
2,district_context,960,19,0,5807,31.84
3,google_trends,14810,8,0,14130,11.93
4,destination_rankings,320,5,0,0,0.00
5,data_dictionary,27,5,0,17,12.59


In [10]:
coverage_summary = pd.DataFrame([
    {
        "dataset": "cleaned_data",
        "geographic_level": "State",
        "locations": datasets["cleaned_data"]["state"].nunique(),
        "start_year": datasets["cleaned_data"]["year"].min(),
        "end_year": datasets["cleaned_data"]["year"].max(),
    },
    {
        "dataset": "state_tourism",
        "geographic_level": "State",
        "locations": datasets["state_tourism"]["state"].nunique(),
        "start_year": datasets["state_tourism"]["year"].min(),
        "end_year": datasets["state_tourism"]["year"].max(),
    },
    {
        "dataset": "district_context",
        "geographic_level": "District",
        "locations": datasets["district_context"]["district"].nunique(),
        "start_year": datasets["district_context"]["year"].min(),
        "end_year": datasets["district_context"]["year"].max(),
    },
    {
        "dataset": "google_trends",
        "geographic_level": "Mixed state/area",
        "locations": datasets["google_trends"]["area"].nunique(),
        "start_year": datasets["google_trends"]["year"].min(),
        "end_year": datasets["google_trends"]["year"].max(),
    },
    {
        "dataset": "destination_rankings",
        "geographic_level": "Destination",
        "locations": datasets["destination_rankings"]["destination"].nunique(),
        "start_year": datasets["destination_rankings"]["year"].min(),
        "end_year": datasets["destination_rankings"]["year"].max(),
    },
])

display(coverage_summary)

,dataset,geographic_level,locations,start_year,end_year
0,cleaned_data,State,16,2019,2025
1,state_tourism,State,16,2019,2025
2,district_context,District,163,2020,2025
3,google_trends,Mixed state/area,167,2018,2025
4,destination_rankings,Destination,164,2024,2025


In [11]:
for name, dataframe in datasets.items():
    missing = (
        dataframe.isna()
        .sum()
        .to_frame("missing_count")
    )

    missing["missing_percentage"] = (
        missing["missing_count"] / len(dataframe) * 100
    ).round(2)

    missing = missing[
        missing["missing_count"] > 0
    ].sort_values("missing_percentage", ascending=False)

    print(f"\n{name.upper()} — missing values")

    if missing.empty:
        print("No missing values.")
    else:
        display(missing)


CLEANED_DATA — missing values


,missing_count,missing_percentage
income_mean,84,75.00
income_median,84,75.00
expenditure_mean,84,75.00
gini,84,75.00
poverty,84,75.00
search_interest_growth_pct,5,4.46



STATE_TOURISM — missing values
No missing values.

DISTRICT_CONTEXT — missing values


,missing_count,missing_percentage
gdp_real_p0,801,83.44
income_median,652,67.92
income_mean,652,67.92
gini,652,67.92
poverty,652,67.92
expenditure_mean,652,67.92
google_trend_yoy_change,229,23.85
ep_ratio,223,23.23
lf,223,23.23
lf_employed,198,20.62



GOOGLE_TRENDS — missing values


,missing_count,missing_percentage
duplicate_column_flag,14130,95.41



DESTINATION_RANKINGS — missing values
No missing values.

DATA_DICTIONARY — missing values


,missing_count,missing_percentage
unit,12,44.44
description,5,18.52
